<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 04 · Data Types and Structures

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the chapter examples in a Colab-ready format so that you
can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time so the objects or plots are
  created in order.
- Add your own cells for experiments or refactorings.
- Use the book text for the surrounding explanation and context.


This chapter builds a modern foundation for working with Python's core data
types and containers. You will learn how to choose between built-in types,
avoid common pitfalls with mutability and copying, and write concise, readable
transformations that prepare data for later chapters on `NumPy`, `pandas`, and
beyond.


# Why Data Types Matter


Picking the right type makes numeric code, container handling, and later
vectorization simpler and less error-prone.


# Built-In Scalar Types


Integers, floats, strings, and booleans are the smallest building blocks you
will combine later.


In [ ]:
# `int` handles arbitrary precision; no overflow in typical finance use.
price = 125


In [ ]:
# `float` represents binary floating-point; see precision notes below.
fee_rate = 0.0015


In [ ]:
# `bool` is a subclass of `int` (`True` equals `1`, `False` equals `0`) but
# should be used for logic, not arithmetic.
is_active = True

In [ ]:
# `str` holds text; always be explicit about encodings when reading/writing
# external data.
label = "EURUSD"

In [ ]:
(price, fee_rate, is_active, label)

## Floating-Point Precision and Safer Alternatives


Binary floating-point can drift; `decimal` and `fractions` help when exactness
matters.


In [ ]:
0.1 + 0.2  # Produces `0.30000000000000004` due to binary representation.

In [ ]:
# Rounding can mask but not remove underlying representation error.
round(0.1 + 0.2, 10)


In [ ]:
from decimal import Decimal, getcontext

In [ ]:
getcontext().prec = 10  # Set precision high enough for your calculations.

In [ ]:
gross = Decimal("100.00")

In [ ]:
fee = gross * Decimal("0.0015")

In [ ]:
# `Decimal` keeps exact decimal semantics for additions and subtractions of
# decimal inputs, and the number of visible decimal places follows the current
# context and operands; you can later quantize to a specific number of places
# when formatting for reports.
net = gross - fee

In [ ]:
net

# Core Containers and Their Roles


Lists, tuples, dicts, and sets solve different problems, and finance code uses
all four.


## Lists and Tuples


Lists are mutable sequences, while tuples are fixed records that make intent
explicit.


In [ ]:
prices = [101.5, 102.0, 100.8]  # A mutable list of floats.

In [ ]:
# An immutable tuple holding a symbol and its price.
latest = ("EURUSD", 1.0832)


In [ ]:
prices.append(103.1)  # Lists support in-place mutation like `append`.

In [ ]:
base, quote = latest  # Tuple unpacking assigns fields by position.

In [ ]:
prices

In [ ]:
base, quote

In [ ]:
# Build a list of spreads in one expression; favor readability over deeply
# nested expressions.
spreads = [ask - bid for bid, ask in [(1.0810, 1.0812), (1.0820, 1.0823)]]

In [ ]:
spreads

## Dictionaries


Dictionaries map keys to values and are ideal for lookup-heavy data such as
parameters and metadata.


In [ ]:
# Initialize with literal syntax for clarity.
quote = {"symbol": "EURUSD", "bid": 1.0810, "ask": 1.0812}

In [ ]:
quote["mid"] = (quote["bid"] + quote["ask"]) / 2  # Add derived fields in place.

In [ ]:
# Dict comprehension builds a mapping from symbol to spread.
spreads = {q["symbol"]: q["ask"] - q["bid"] for q in [quote]}

In [ ]:
quote

In [ ]:
spreads

In [ ]:
# Returns a default when the key is absent instead of raising.
mid = quote.get("mid", float("nan"))


In [ ]:
mid

## Sets


Sets support fast membership checks and duplicate removal when order does not
matter.


In [ ]:
# Literal creates a set; duplicates collapse automatically.
symbols = {"AAPL", "MSFT", "AAPL"}


In [ ]:
# Casting from an iterable also removes duplicates.
unique_symbols = set(["EURUSD", "USDJPY", "EURUSD"])

In [ ]:
"AAPL" in symbols  # Membership tests are O(1) on average.

In [ ]:
symbols

In [ ]:
unique_symbols

# Copying and Mutability


Aliasing and shallow copies can surprise you when multiple names point to the
same object.


In [ ]:
import copy

In [ ]:
positions = [{"symbol": "AAPL", "qty": 10}]

In [ ]:
# Shallow copy: inner dict is shared; later mutations propagate.
shallow = list(positions)


In [ ]:
deep = copy.deepcopy(positions)  # Deep copy: inner dict is independent.

In [ ]:
# After this line, `shallow[0]["qty"]` is 20; `deep[0]["qty"]` remains 10.
positions[0]["qty"] = 20


In [ ]:
shallow

In [ ]:
deep

# Handy Built-Ins for Iteration


Built-ins like `zip`, `enumerate`, `sorted`, and `sum` simplify common data-
wrangling loops.


In [ ]:
symbols = ["AAPL", "MSFT", "GOOG"]

In [ ]:
prices = [180.0, 350.0, 140.0]

In [ ]:
# `zip` pairs sequences; `enumerate` adds a counter starting at 1.
for i, (sym, px) in enumerate(zip(symbols, prices), start=1):
    print(f"{i}. {sym} -> {px}")

# Control Flow Essentials


Conditionals and loops let you shape program behavior before you move to
vectorized code.


## if / elif / else


Branching is how you express the simple decisions that appear throughout data
cleaning and reporting.


In [ ]:
price = 105

In [ ]:
threshold = 100

In [ ]:
# First matching branch executes; subsequent branches are skipped.
if price > threshold:
    status = "above"
# Use `elif` for mutually exclusive conditions; keep the number of branches
# small for readability.
elif price == threshold:
    status = "equal"
else:
    status = "below"

In [ ]:
status

## for loops


A `for` loop is the clearest way to repeat a calculation across a sequence.


In [ ]:
symbols = ["AAPL", "MSFT", "GOOG"]

In [ ]:
# Iterating over values keeps the loop concise; avoid manual indexing when not
# needed.
for sym in symbols:
    print(sym.lower())

In [ ]:
numbers = [10, 20, 30]

In [ ]:
# Use `range` for index-based loops; keep the body small and consider
# `enumerate` for readability.
for i in range(len(numbers)):
    print(i, numbers[i])

## while loops


Use a `while` loop when the number of iterations depends on a condition, not
on collection length.


In [ ]:
n = 3

In [ ]:
total = 0

In [ ]:
while n > 0:  # Condition is checked before each iteration.
    total += n
    n -= 1  # Update the loop variable to guarantee termination.

In [ ]:
total

## break and continue


`break` exits a loop early, and `continue` skips work for the current item.


In [ ]:
data = [1, -1, 2, 3]

In [ ]:
total = 0

In [ ]:
for x in data:
    if x < 0:  # Guard condition to filter values.
        continue  # Skip negative values without exiting the whole loop.
    total += x

In [ ]:
total

## any and all for predicates


`any` and `all` let you test collections without writing manual accumulator
code.


In [ ]:
values = [1, 2, 3, -5]

In [ ]:
# `any` returns True if any element matches the predicate.
any(v < 0 for v in values)


In [ ]:
# `all` returns True only if every element matches the predicate.
all(v > 0 for v in values)


## Exceptions: try / except / else / finally


Exception handling keeps failures local and makes cleanup explicit.


In [ ]:
data = {"close": "not-a-number"}

In [ ]:
from decimal import Decimal, InvalidOperation

In [ ]:
try:  # Keep the `try` body minimal; only the code that may raise goes inside.
    # Operation that might fail (missing key or bad conversion).
    price = Decimal(data["close"])
except KeyError:  # Handle missing key explicitly.
    price = Decimal("NaN")
except InvalidOperation:  # Handle invalid numeric conversion explicitly.
    price = Decimal("NaN")
# `else` runs only if no exception was raised; place success-path logic here.
else:
    price = price.quantize(Decimal("0.0001"))
finally:  # `finally` always runs; use it for cleanup or bookkeeping.
    status = "done"

In [ ]:
price, status

# Comprehensions


Comprehensions express common transformations in a compact, readable form.


## List comprehensions


List comprehensions replace short append loops when you are building a new
list.


In [ ]:
values = [1, 2, 3, 4]

In [ ]:
# Build a new list by applying an expression to each element.
squares = [v * v for v in values]


In [ ]:
# Add an inline filter to keep only even numbers.
evens = [v for v in values if v % 2 == 0]


In [ ]:
squares

In [ ]:
evens

## Dict comprehensions


Dict comprehensions build keyed lookup tables from existing data.


In [ ]:
quotes = [
    {"symbol": "EURUSD", "bid": 1.0810, "ask": 1.0812},
    {"symbol": "USDJPY", "bid": 149.8, "ask": 149.82},
]

In [ ]:
# Build a mapping from symbol to mid-price in one pass.
mids = {q["symbol"]: (q["bid"] + q["ask"]) / 2 for q in quotes}

In [ ]:
mids

## Set comprehensions


Set comprehensions deduplicate transformed values in one pass.


In [ ]:
raw_symbols = ["aapl", "AAPL", "msft", "MSFT"]

In [ ]:
# Normalize case and remove duplicates in a single pass.
canonical = {s.upper() for s in raw_symbols}


In [ ]:
canonical

# Type Hints for Clarity


Type hints document intent and make it easier to catch mismatches in larger
codebases.


In [ ]:
from collections.abc import Iterable

In [ ]:
def gross_returns(prices: Iterable[float]) -> list[float]:
    """Compute gross returns from a price series."""
    # Convert to list once to allow indexing multiple times.
    prices = list(prices)
    # List comprehension computes successive returns; type hints make expected
    # input/output clear, and importing `Iterable` from
    # `collections.abc` matches
    # modern Python style.
    return [(prices[i] / prices[i - 1]) - 1 for i in range(1, len(prices))]

In [ ]:
gross_returns([100, 105, 110])

# Where We Are Heading Next


Chapter 5 applies these language tools to array-based numerical work with
`NumPy`.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
